# Tesis - MDM UBA - 2026

**Tariff classification using NLP**

By Santiago Tedoldi

# Testing and using final model


## Test sample

In [70]:
# Dependencies
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Loading model training config

In [71]:
with open('results/distilbert/fft_final/final_model/training_config.json', 'r') as file:
    # The json.load() function reads the file and returns a Python object
    model_cofig = json.load(file)

with open('results/distilbert/fft_final/labels/labels_dict_FINAL_DBERT_fft_GOODS_DESCRIPTION_HS04_seed32.json', 'r') as file:
    # The json.load() function reads the file and returns a Python object
    labels_dict = json.load(file)

label2id = labels_dict["label2id"]
id2label = labels_dict["id2label"]
id2label = {int(k): v for k, v in id2label.items()}

model_cofig

{'run_name': 'FINAL_DBERT_fft_GOODS_DESCRIPTION_HS04_seed32',
 'train_type': 'fft',
 'text_col': 'GOODS_DESCRIPTION',
 'target_col': 'HS04',
 'max_length': 300,
 'batch_size': 128,
 'lr': 5e-05,
 'test_fraction': 0.01,
 'seed': 32,
 'max_epochs': 5,
 'early_stopping': True,
 'monitor': 'val_loss',
 'patience': 3,
 'min_delta': 0.0,
 'warmup_epochs': 1,
 'fine_tune': True,
 'n_finetune_layers': 0}

### Test dataset

In [72]:
data_type = {'Description': str,
             'True Label': str,
             'Top1': str,
             'Top2': str,
             'Top3': str,
             'Top4': str,
             'Top5': str,
             }

test = pd.read_csv('results/distilbert/fft_final/final_val_predictions.csv', index_col=0,
                   dtype=data_type)
test.head()

,Description,True Label,Top1,Proba Top1,Top2,Proba Top2,Top3,Proba Top3,Top4,Proba Top4,Top5,Proba Top5
354196,sterilizing cabinet,8419,9402,0.265925,9403,0.247374,9018,0.227738,8419,0.099247,8537,0.027063
155635,PERFUMES-LOVE INTETION,3303,3303,0.942016,3307,0.036050,3302,0.015176,3301,0.001438,3304,0.000576
134060,High voltage insulating tape,3920,8546,0.445084,3919,0.234640,5906,0.089700,3920,0.048213,3921,0.029814
114018,BEARING 6805 ZZ BRAND KG,8482,8482,0.995477,8483,0.000293,7319,0.000043,8711,0.000042,7009,0.000021
264869,USED SOFA 3SEATER,9403,9403,0.621796,9401,0.366761,9404,0.001297,8715,0.000630,9402,0.000514


In [73]:
len(test['True Label'].unique())

507

## DistilBERT application


In [74]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
# from transformers import Trainer, TrainingArguments, TrainerCallback
import torch


class TokenizedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }
    
    
class HSClassifier(nn.Module):
    def __init__(self,
                 n_classes: int,
                 fine_tune: bool = False,
                 n_finetune_layers: int = 0):
        """
        Args:
          n_classes:      number of target classes
          fine_tune:      if True, you’ll unfreeze either all or the last layers
          n_finetune_layers:
                          • =0 (default) → if fine_tune=True, unfreeze *all* DistilBERT layers  
                          • >0             → unfreeze only that many of the *last* transformer blocks  
                          • ignored if fine_tune=False (encoder stays fully frozen)
        """
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # Freeze everything by default
        for param in self.distilbert.parameters():
            param.requires_grad = False

        # If fine_tune, decide what to unfreeze
        if fine_tune:
            if n_finetune_layers > 0:
                # Unfreeze only the last `n_finetune_layers` transformer blocks
                for block in self.distilbert.transformer.layer[-n_finetune_layers:]:
                    for param in block.parameters():
                        param.requires_grad = True
            else:
                # n_finetune_layers == 0 → unfreeze *all* DistilBERT params
                for param in self.distilbert.parameters():
                    param.requires_grad = True

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.distilbert.config.hidden_size, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, n_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # Take <CLS> token representation
        logits = self.classifier(hidden_state)
        return logits
    

class HSDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length, text_col='GOODS_DESCRIPTION',
                 label_col='HS04'):
        self.descriptions = dataframe[text_col].tolist()
        self.labels = dataframe[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.descriptions[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': self.labels[idx],
            'description': self.descriptions[idx]
        }
        return item


def predict_and_evaluate(
    model, tokenizer, unseen_sample, text_col, label_col, id2label,
    max_length=128, device='cpu', batch_size=32,
):
    """
    Predicts the top 5 classes and their probabilities for an unseen sample using a given model and tokenizer.
    Calculates the accuracy for top 1 to top 5 predictions.
    Uses a DataLoader for GPU memory efficiency.
    """
    # Dataset & DataLoader
    dataset = HSDataset(unseen_sample, tokenizer, max_length, text_col, label_col)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.to(device)
    model.eval()

    all_top5_predicted_labels = []
    all_top5_predicted_probs = []
    all_true_labels = []
    all_descriptions = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probabilities = F.softmax(outputs, dim=1)
            top5_probs, top5_preds = torch.topk(probabilities, 5, dim=1)

            # Convert predictions and probabilities to lists
            for i in range(top5_preds.size(0)):
                pred_labels = [id2label[idx.item()] for idx in top5_preds[i]]
                pred_probs = [prob.item() for prob in top5_probs[i]]
                all_top5_predicted_labels.append(pred_labels)
                all_top5_predicted_probs.append(pred_probs)
            
            all_true_labels.extend(batch['label'])
            all_descriptions.extend(batch['description'])

    # Accuracy calculations
    accuracy_top1 = sum([
        all_true_labels[i] == all_top5_predicted_labels[i][0]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top2 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:2]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top3 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:3]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top4 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:4]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top5 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:5]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)

    print(f"Accuracy Top-1: {accuracy_top1*100:.2f} %")
    print(f"Accuracy Top-2: {accuracy_top2*100:.2f} %")
    print(f"Accuracy Top-3: {accuracy_top3*100:.2f} %")
    print(f"Accuracy Top-4: {accuracy_top4*100:.2f} %")
    print(f"Accuracy Top-5: {accuracy_top5*100:.2f} %")

    # Results DataFrame
    results = pd.DataFrame({
        'Description': all_descriptions,
        'True Label': all_true_labels,
        'Top1': [labels[0] for labels in all_top5_predicted_labels],
        'Proba Top1': [probs[0] for probs in all_top5_predicted_probs],
        'Top2': [labels[1] for labels in all_top5_predicted_labels],
        'Proba Top2': [probs[1] for probs in all_top5_predicted_probs],
        'Top3': [labels[2] for labels in all_top5_predicted_labels],
        'Proba Top3': [probs[2] for probs in all_top5_predicted_probs],
        'Top4': [labels[3] for labels in all_top5_predicted_labels],
        'Proba Top4': [probs[3] for probs in all_top5_predicted_probs],
        'Top5': [labels[4] for labels in all_top5_predicted_labels],
        'Proba Top5': [probs[4] for probs in all_top5_predicted_probs],
    })
    results.set_index(unseen_sample.index, inplace=True)

    return results


In [75]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2677 entries, 354196 to 38217
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Description  2677 non-null   object 
 1   True Label   2677 non-null   object 
 2   Top1         2677 non-null   object 
 3   Proba Top1   2677 non-null   float64
 4   Top2         2677 non-null   object 
 5   Proba Top2   2677 non-null   float64
 6   Top3         2677 non-null   object 
 7   Proba Top3   2677 non-null   float64
 8   Top4         2677 non-null   object 
 9   Proba Top4   2677 non-null   float64
 10  Top5         2677 non-null   object 
 11  Proba Top5   2677 non-null   float64
dtypes: float64(5), object(7)
memory usage: 271.9+ KB


In [76]:
# Load the tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Tokenizing val data
test_encodings = tokenizer(
    list(test['Description']), 
    truncation=True,
    padding='max_length',
    max_length=model_cofig["max_length"],
    return_tensors='pt'
)

test_labels = torch.tensor([label2id[str(lbl)] for lbl in test['True Label']])

# test_dataset = TokenizedDataset(test_encodings, test_labels)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0)

In [78]:
model_path = 'results/distilbert/fft_final/final_model/pytorch_model.bin'

model_final = HSClassifier(n_classes=len(label2id), fine_tune=model_cofig['fine_tune'])

state_dict = torch.load(model_path, map_location="cpu")
model_final.load_state_dict(state_dict)
model_final.eval()

C:\Users\santt\AppData\Local\Temp\ipykernel_23964\777545881.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")


HSClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Lin

In [79]:
torch.cuda.empty_cache()

In [86]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Testing on {device}')

test_reprocced = predict_and_evaluate(model_final, 
                                         tokenizer, 
                                         test, 
                                         text_col='Description',
                                         label_col='True Label',
                                         id2label=id2label, 
                                         max_length=model_cofig["max_length"], 
                                         device=device)

Testing on cuda
Accuracy Top-1: 66.72 %
Accuracy Top-2: 75.46 %
Accuracy Top-3: 79.68 %
Accuracy Top-4: 82.03 %
Accuracy Top-5: 83.83 %


In [87]:
test_reprocced

,Description,True Label,Top1,Proba Top1,Top2,Proba Top2,Top3,Proba Top3,Top4,Proba Top4,Top5,Proba Top5
354196,sterilizing cabinet,8419,9402,0.265926,9403,0.247373,9018,0.227738,8419,0.099247,8537,0.027063
155635,PERFUMES-LOVE INTETION,3303,3303,0.942016,3307,0.036050,3302,0.015176,3301,0.001438,3304,0.000576
134060,High voltage insulating tape,3920,8546,0.445084,3919,0.234640,5906,0.089700,3920,0.048213,3921,0.029814
114018,BEARING 6805 ZZ BRAND KG,8482,8482,0.995477,8483,0.000293,7319,0.000043,8711,0.000042,7009,0.000021
264869,USED SOFA 3SEATER,9403,9403,0.621797,9401,0.366760,9404,0.001297,8715,0.000630,9402,0.000514
...,...,...,...,...,...,...,...,...,...,...,...,...
59522,"WASHER, FLAT: NOMINAL SIZE: M22; MATERIAL: STEEL",7318,7318,0.981964,7415,0.015497,8450,0.000313,4016,0.000112,7320,0.000092
2697,ZEOLITH UNIT E080 AUTOTROL (2PCS),3006,3824,0.644254,3006,0.083513,3808,0.023826,3822,0.017465,3506,0.015863
476065,ADIDAS VL COURT 2.0 JN 19,6402,6404,0.378602,6402,0.343914,6114,0.050933,6403,0.048458,6204,0.025390
98085,AIR FILTER MATERIALS,8421,8421,0.995163,5911,0.000568,9002,0.000530,7019,0.000445,4823,0.000151


In [88]:
test_reprocced.sort_values(by='Proba Top1', ascending=False)

,Description,True Label,Top1,Proba Top1,Top2,Proba Top2,Top3,Proba Top3,Top4,Proba Top4,Top5,Proba Top5
27487,SL ZEUS ANTRACITA (PRC) 60 X 60 STD,6907,6907,0.999904,7016,0.000007,1904,0.000006,6904,0.000005,5702,0.000004
136893,STRYDOL MEGAMILE SUPREME 15W40 (210L,2710,2710,0.999897,3819,0.000072,3403,0.000022,3820,0.000003,2827,0.000002
185712,USED SCANIA TRACTOR 4 X 2,8701,8701,0.999853,8429,0.000007,8711,0.000006,8426,0.000005,8704,0.000004
327352,USED SCANIA R470 4X2 MANUAL TRACTOR UNIT,8701,8701,0.999849,8429,0.000018,8711,0.000009,8426,0.000008,8704,0.000005
471018,VAN LOVEREN RIVER ROSE BLANC DE NOIR RED MUSCA...,2204,2204,0.999828,2203,0.000039,2206,0.000021,0806,0.000017,2208,0.000013
...,...,...,...,...,...,...,...,...,...,...,...,...
271589,SOULDER,6217,8421,0.043645,8714,0.032468,3808,0.031463,7323,0.028452,8516,0.025421
275993,BUCKECTS,6911,9506,0.043034,7318,0.033154,9403,0.032755,6304,0.028558,3926,0.025739
172847,CABAS,4202,6203,0.032557,2710,0.028059,8544,0.026147,8714,0.025611,6114,0.019470
436501,COLGANTE DE CUERO,7117,6204,0.028127,2103,0.025122,6907,0.023948,8509,0.020929,8516,0.020739


In [ ]:
######

## Module-based inference usage
Reusable inference using `hs04_distilbert_inference.py`


In [121]:
from hs04_distilbert_inference import HS04DistilBERTPredictor

predictor = HS04DistilBERTPredictor(
    model_path='results/distilbert/fft_final/final_model/pytorch_model.bin',
    config_path='results/distilbert/fft_final/final_model/training_config.json',
    labels_path='results/distilbert/fft_final/labels/labels_dict_FINAL_DBERT_fft_GOODS_DESCRIPTION_HS04_seed32.json',
)

print(f'Device: {predictor.device} | max_length: {predictor.max_length}')


c:\Users\santt\Desktop\DataMining_UBA\4-taller-1\tt1_clasi\hs04_distilbert_inference.py:109: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(self.model

Device: cuda | max_length: 300


In [122]:
example_text = test.iloc[0]['Description']
single_pred = predictor.predict(example_text, top_k=5)
single_pred


{'Description': 'sterilizing cabinet',
 'Top1': '9402',
 'Proba Top1': 0.2659248113632202,
 'predictions': [{'label': '9402', 'probability': 0.2659248113632202},
  {'label': '9403', 'probability': 0.24737350642681122},
  {'label': '9018', 'probability': 0.22773753106594086},
  {'label': '8419', 'probability': 0.099247045814991},
  {'label': '8537', 'probability': 0.0270627923309803}]}

In [124]:
example_text = "samsung smart phone S25+"
single_pred = predictor.predict(example_text, top_k=5)
single_pred

{'Description': 'samsung smart phone S25+',
 'Top1': '8517',
 'Proba Top1': 0.995586633682251,
 'predictions': [{'label': '8517', 'probability': 0.995586633682251},
  {'label': '8471', 'probability': 0.0009033721289597452},
  {'label': '8525', 'probability': 0.00039607175858691335},
  {'label': '8470', 'probability': 0.00017427954298909754},
  {'label': '8518', 'probability': 0.0001606932346476242}]}

In [123]:
batch_texts = test['Description'].head(5).tolist()
batch_preds = predictor.predict_batch(batch_texts, top_k=5, batch_size=8)
batch_preds


,Description,Top1,Proba Top1,Top2,Proba Top2,Top3,Proba Top3,Top4,Proba Top4,Top5,Proba Top5
0,sterilizing cabinet,9402,0.265925,9403,0.247374,9018,0.227737,8419,0.099247,8537,0.027063
1,PERFUMES-LOVE INTETION,3303,0.942016,3307,0.036050,3302,0.015176,3301,0.001438,3304,0.000576
2,High voltage insulating tape,8546,0.445085,3919,0.234640,5906,0.089700,3920,0.048213,3921,0.029814
3,BEARING 6805 ZZ BRAND KG,8482,0.995477,8483,0.000293,7319,0.000043,8711,0.000042,7009,0.000021
4,USED SOFA 3SEATER,9403,0.621797,9401,0.366760,9404,0.001297,8715,0.000630,9402,0.000514


In [125]:
batch_texts = [
    "samsung smart phone S25+",
    "electric motor 50hp GE"
]
batch_preds = predictor.predict_batch(batch_texts, top_k=5, batch_size=8)
batch_preds

,Description,Top1,Proba Top1,Top2,Proba Top2,Top3,Proba Top3,Top4,Proba Top4,Top5,Proba Top5
0,samsung smart phone S25+,8517,0.995587,8471,0.000903,8525,0.000396,8470,0.000174,8518,0.000161
1,electric motor 50hp GE,8501,0.978849,8511,0.004645,8412,0.002804,8711,0.001487,8407,0.001360
